# Project 1

Welcome to the first NLP1 project. The goal of the first tutorial was to get you familiar with text segmentation and simple text classification for topic detection.
Given the skills you have developed in the first week, you will now be applying them within a different dataset and task.

We are recommending a set of tasks and associated datasets; using those, you will be experimenting on problems introduced during the tutorials. You will be applying hands-on the experiments you have already explored in the last tutorial, and taking a deeper dive: extending your experiments, trying different set ups, possibly considering different evaluation techniques and analysing errors. 

The week after the deadline, your TA will perform a short oral assessment of the skills and knowledge you acquired (make sure to check the official schedule for this), probing you to answer questions about what you learnt, about what you did and about why you did the things you did (as opposed to some reasonable alternatives) as well as pros and cons of your method, and so on. 

As the tutorials will be building upon your knowledge throughout the weeks, the task and dataset you choose now will be the ones you use in projects 2 and 3; and the ones you will be finally writing about in your final blog post.  

## Get to work!

The next section has a checklist to help you not lose track of what needs to be done. Here we have a more informal guide, with prompts and questions for you. 


1. [Choose a dataset](http://nlp-illc.github.io/nlp1/). Spend some time understanding how the dataset looks and some statistics of it (e.g. how many instances of each classification label we have?)
2. Remember what we learnt during the tutorial: to develop any classifier of the kind we saw, we needed a way to segment text. Do you want to use the tokeniser we have developed? Do you need to train a new one? If you do, what hyperparameters shall you use? What data shall you use to train it? Consider that your dataset may be too small to train a tokeniser! Think whether you will need a larger resource of similar nature to train a tokeniser.
3. Start thinking how to represent your documents into features. We have experimented with token occurences and frequencies. Can you think of any other ways, possibly? 
4. We had experimented with Logistic Regression and a shallow neural network. You can experiment with and assess different architectures now and their hyper parameters (e.g. add layers, possibly change the activation functions, drop out rates).
5. Evaluate your models! Was the evaluation method adequate for your new experimental setup? Do you want to assess other aspects of performance?
6. Analyse the various models you assessed and try to decide on which is the best.

Be mindful that you will be using your experiments to write up your final blog, so make sure they are clean and neat!

You can write your own code and you can reuse code from the tutorial and its packages or from other sources. You may use coding assistants if you like, but use it mindfully as you take full responsibility for this kind of (semi-)automated coding. Also, keep a record of the kinds of things you delegate to any form of assistant, as you will be asked to disclose this in your blog post (not because we will pass a value judgement on it, but because it is a widespread and good practice to document such things).

You may use jupyter notebooks or python files, it's up to you as we will not directly assess your code. But, once again, be organised, as we will ask you to share a reproducible version of your experiments along with your blog post (again, because this is how real research is publicised). 

If you are unsure about anything, ask your TA! They are always happy to help you! Happy coding! :)

## Checklist

What **we expect you to do**:

- [ ] Compare different systems of text classification for your choice of data/task.
- Your comparisons should cover:
    - [ ] the two representation spaces in the tutorial (token indicators and bag-of-words) and an additional one of your choice.
    - [ ] the two kinds of architectures in the tutorial (log-linear and single-hidden-layer MLP) and an additional one of your choice (such as a modification of the MLP).
- [ ] Your comparisons must be based on cross-validation and a relevant metric (accuracy or else).
- [ ] You should make use of Bayesian estimation to help you analyse results in relation to a region of practical equivalence (ROPE), and you can motivate the ROPE size yourself.

What you need to **bring to the oral assessment**:
* You will be asked questions about the points above, you can bring a slide deck with up to five slides (one for each of the following topics _data and metrics_, _features_, _architectures_, _results_, _extra_); you _will not_ give a presentation, rather, these slides are to be thought of as "backup slides" that you can use to guide your answers to the questions the TA will ask you.
* You may also be asked questions about the tutorial, for those, you don't need to prepare any deliverable (a good way to prepare is to work through the tutorial and reflect on the questions there).

What you should **keep in mind** given that you will eventually write a blog post about your work throughout the course:
* Organise your code;
* Summarise your motivations and findings;
* Document your sources and any form of AI-assistance you use.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import part1
from datasets import load_dataset

dataset = load_dataset("google-research-datasets/paws", "labeled_final")
paws_outcome_space = part1.clslib.NamedClasses(['not paraphrase', 'paraphrase'])


## WE use the PAWS dataset
paws consist of a dataset of two sentences with a binary classsification with the question if two of the sentences are a pharaphrase of the other.


# Dataset statistics and examples

In [2]:
#An example of a sentence pair from our training corpus
print("Sentence 1:", dataset["train"]["sentence1"][0])
print("Sentence 2:", dataset["train"]["sentence2"][0])
print("Gold label (id):", dataset["train"]["label"][0])

Sentence 1: In Paris , in October 1560 , he secretly met the English ambassador , Nicolas Throckmorton , asking him for a passport to return to England through Scotland .
Sentence 2: In October 1560 , he secretly met with the English ambassador , Nicolas Throckmorton , in Paris , and asked him for a passport to return to Scotland through England .
Gold label (id): 0


In [3]:
from collections import Counter
from statistics import mean, median

def count_labels(data):
    """
    count the number of data points for weach of the binary labels in the dataset
    """
    label_counts = Counter(data["label"])
    return label_counts

def print_precentages(label_counts, total):
    """
    print the percentage of each label in the dataset
    """
    for label, count in label_counts.items():
        percentage = (count / total) * 100
        print(f"Label {label}: {percentage }%")

def entence_length_stats(data):
    """
    compute the mean and median length of the sentences in the dataset
    """
    sentence_lengths = [(len(s1.split()), len(s2.split()), label) for s1, s2, label in zip(data["sentence1"], data["sentence2"], data["label"])]
    mean_length_s1 = mean(length[0] for length in sentence_lengths)
    mean_length_s2 = mean(length[1] for length in sentence_lengths)
    median_length_s1 = median(length[0] for length in sentence_lengths)
    median_length_s2 = median(length[1] for length in sentence_lengths)
    print(f"Mean length of sentence 1 (based on spaces): {mean_length_s1}")
    print(f"Mean length of sentence 2 (based on spaces): {mean_length_s2}")
    print(f"Median length of sentence 1 (based on spaces): {median_length_s1}")
    print(f"Median length of sentence 2 (based on spaces): {median_length_s2}")
    return mean_length_s1, mean_length_s2, median_length_s1, median_length_s2

print("Training set label counts:")
train_label_counts = count_labels(dataset["train"])
print_precentages(train_label_counts, len(dataset["train"]))
_ = entence_length_stats(dataset["train"])

Training set label counts:
Label 0: 55.81263537175361%
Label 1: 44.18736462824639%
Mean length of sentence 1 (based on spaces): 21.340782575251513
Mean length of sentence 2 (based on spaces): 21.337948624521772
Median length of sentence 1 (based on spaces): 21
Median length of sentence 2 (based on spaces): 21


In [4]:
import part1

In [5]:
# PAWS has no "text" column: use both sentences of every pair as the corpus
corpus = list(dataset["train"]["sentence1"]) 
print("datapoints: ", len(corpus))
print()


datapoints:  49401



In [6]:
vocab_size = 5000
min_frequency = 2
tokenizer = part1.experiment.train_tokenizer(corpus, vocab_size, min_frequency)

# tokenize the data
1) Token indicators (like the tutorial)
2) Bag of words (home made)
3) UNKOWN

In [7]:
#Token indicator code
import numpy as np  
from tokenizers import Tokenizer
class Encoder:

    def __len__(self) -> int:
        """Dimensionality of the representation space"""
        raise NotImplementedError("Concrete implementations will be discussed later")

    def __call__(self, x: str) -> 'np.array':
        """Represent x as a D-dimensional vector of real numbers"""
        raise NotImplementedError("Concrete implementations will be discussed later")
        
class TokenIndicator(Encoder):

    def __init__(self, tokenizer: Tokenizer):
        self.tokenizer = tokenizer
        
    def __len__(self) -> int:
        """Dimensionality of the representation space"""
        return len(tokenizer.get_vocab())

    def __call__(self, x: str) -> 'np.array':
        """Represent x as a D-dimensional vector of real numbers"""
        unique_token_ids = list(set(self.tokenizer.encode(x).ids))

        # A 0 vector with the same length as the vocabulary except in positions corresponding 
        # to tokens appearing in the document
        vector = np.zeros(len(self))
        vector[unique_token_ids] = 1
        return vector

data_token_indicator = TokenIndicator(tokenizer)


In [8]:
# bag of words representation
class BagOfWords(Encoder):
    def __init__(self, tokenizer: Tokenizer):
        self.tokenizer = tokenizer

    def __len__(self) -> int:
        """Dimensionality of the representation space"""
        return len(self.tokenizer.get_vocab())

    def __call__(self, x: str) -> 'np.array':
        """Represent x as a D-dimensional vector of real numbers"""
        encoded = self.tokenizer.encode(x)
        vector = np.zeros(len(self))
        for token_id in encoded.ids:
            vector[token_id] += 1
        return vector

data_bag_of_words = BagOfWords(tokenizer)

In [9]:
import numpy as np
import torch
from pathlib import Path
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel

BERT_NAME = "bert-base-uncased"
CACHE = Path(f"bert_feats_{BERT_NAME.replace('/', '_')}.npy")
device = "cuda" if torch.cuda.is_available() else "cpu"

bert_tok = AutoTokenizer.from_pretrained(BERT_NAME)
bert = AutoModel.from_pretrained(BERT_NAME).to(device).eval()
@torch.no_grad()

def bert_pair(s1, s2, batch_size=64):
    """Encode each (s1, s2) pair jointly as [CLS] s1 [SEP] s2 [SEP].
    Returns [N, 1536]: the [CLS] vector concatenated with the mean-pooled token vectors."""
    feats = []
    for i in tqdm(range(0, len(s1), batch_size)):
        enc = bert_tok(s1[i:i+batch_size], s2[i:i+batch_size],
                       padding=True, truncation=True, max_length=128,
                       return_tensors="pt").to(device)
        h = bert(**enc).last_hidden_state
        cls = h[:, 0]
        mask = enc["attention_mask"].unsqueeze(-1)
        mean = (h* mask).sum(1) / mask.sum(1)
        feats.append(torch.cat([cls, mean], dim=-1).cpu().numpy())
    return np.vstack(feats).astype(np.float32)

s1_all = list(dataset["train"]["sentence1"])
s2_all = list(dataset["train"]["sentence2"])

if CACHE.exists():
    X_bert = np.load(CACHE)
else:
    X_bert = bert_pair(s1_all, s2_all)
    np.save(CACHE, X_bert)


In [10]:
#example of both tkenizers
print("Sentence 1:", dataset["train"]["sentence1"][0])
print("Token Indicator representation:", data_token_indicator(dataset["train"]["sentence1"][0]))
print("total number of tokens in the sentence:", sum(data_token_indicator(dataset["train"]["sentence1"][0])))
#example of the representation of a sentence using the BagOfWords
print("Sentence 1:", dataset["train"]["sentence1"][0])
print("Bag of Words representation:", data_bag_of_words(dataset["train"]["sentence1"][0]))
print("total number of tokens in the sentence:", sum(data_bag_of_words(dataset["train"]["sentence1"][0])))
# example of the representation of a sentence pair using BERT
print("Sentence 1:", dataset["train"]["sentence1"][0])
print("Sentence 2:", dataset["train"]["sentence2"][0])
print("BERT representation:", len(X_bert[0]), "dimensionality:", X_bert.shape[1])


Sentence 1: In Paris , in October 1560 , he secretly met the English ambassador , Nicolas Throckmorton , asking him for a passport to return to England through Scotland .
Token Indicator representation: [0. 0. 0. ... 0. 0. 0.]
total number of tokens in the sentence: 37.0
Sentence 1: In Paris , in October 1560 , he secretly met the English ambassador , Nicolas Throckmorton , asking him for a passport to return to England through Scotland .
Bag of Words representation: [0. 0. 0. ... 0. 0. 0.]
total number of tokens in the sentence: 41.0
Sentence 1: In Paris , in October 1560 , he secretly met the English ambassador , Nicolas Throckmorton , asking him for a passport to return to England through Scotland .
Sentence 2: In October 1560 , he secretly met with the English ambassador , Nicolas Throckmorton , in Paris , and asked him for a passport to return to Scotland through England .
BERT representation: 1536 dimensionality: 1536


# Train calssiefieres
1) Log linear like in the tutorial
2) the Mlp


In [11]:
import part1
my_seeds = [0] # [0, 3, 5] # add more seeds here for multiple runs of cross validation, eg. [0, 3, 5]  
num_folds = 5  # more makes each cross validation run more statistically informative, but it can get slow
my_cross_validation_sets = []

for seed in my_seeds:  
    my_cross_validation_sets.append(
        part1.experiment.cross_validation_split(
            # we pair the texts with their labels
            pairs=list(zip(dataset["train"]["sentence1"], dataset["train"]["sentence2"], dataset["train"]["label"])), 
            
            k=num_folds,
            seed=seed
        )
    )

# loggistic 
WE have to rebuild the logistic regression from the tutorial since we have two input sentences so the fit is not correct. We work our own magic on it to make it work for two sentences.

WE have the adapted dunction from the turorial, we concate the two encoded verctors so it can be used as input.

In [12]:
from scipy.sparse import csr_matrix
from sklearn.linear_model import LogisticRegression
import pickle
from tqdm.auto import tqdm

def load(path):
    """
    Load a logistic model from {path}.pickle and return it.
    """
    with open(f"{path}.pickle", 'rb') as f:
        logistic_model = pickle.load(f)
    return logistic_model

def fit(training_set, encoder: 'Encoder', save_to=None, max_iter=100, n_jobs=-1):
    """
    Constructs, trains and returns a softmax regressor.
    
    training_set: list of text-label pairs (a text is a string, a label is a 0-based integer)
    encoder: an instance of the Encoder class (such as TokenIndicator)    
    save_to: (str) if given, save the model to {save_to}.pickle
    """
    # Convert the training and validation sets to vectors
    # we get two senteces for each data point, so we make it a vector [sentence1, sentence2]
    training_vectors = [np.concatenate([encoder(text[0]), encoder(text[1])]) for text in training_set]
    
    training_labels = [label for _, _, label in training_set]

    #To train the model more efficiently, we transform the training vectors to a sparse matrix
    X_train_sparse = csr_matrix(training_vectors)

    # Train the logistic model
    logistic_model = LogisticRegression(solver="saga", max_iter=max_iter, n_jobs=n_jobs) #sparse matrix frinedly solver 
    logistic_model.fit(X_train_sparse, training_labels)

    if save_to is not None:        
        with open(f"{save_to}.pickle", 'wb') as f:
            pickle.dump(logistic_model, f, protocol=pickle.HIGHEST_PROTOCOL)
        
    return logistic_model        

def predict(texts, model, encoder: 'Encoder'):
    """
    Predicts labels (via the most probable class decision rule) for a collection of texts. 
    (scikit-learn hard-codes the most probable class as the decision rule of choice)
    
    texts: a list of texts, each a string
    model: a trained instance of sklearn's LogisticRegression
    encoder: the Encoder used for in the model's training
    """
    validation_vectors = [encoder(text) for text in texts]
    X_val_sparse = csr_matrix(validation_vectors)
    predictions = model.predict(X_val_sparse)
    return predictions

def predict_probs(texts, model, encoder: 'Encoder'):
    """
    Predicts labels (via the most probable class decision rule) for a collection of texts. 
    (scikit-learn hard-codes the most probable class as the decision rule of choice)
    
    texts: a list of texts, each a string
    model: a trained instance of sklearn's LogisticRegression
    encoder: the Encoder used for in the model's training
    """
    validation_vectors = [encoder(text) for text in texts]
    X_val_sparse = csr_matrix(validation_vectors)
    probs = model.predict_proba(X_val_sparse)
    return probs


In [13]:
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from scipy.sparse import csr_matrix
import numpy as np
from collections import defaultdict

fold_results = []
models = {}
#Cross validation is a list of dictionaries, where each dictionary contains the training and validation sets for a fold
def cross_validation_train_logistic(data_token_indicator, my_cross_validation_sets, num_folds):
    """
    Train a logistic regression model on each fold of the cross validation sets and evaluate it on the validation set.
    """
    for index, cross_validation in my_cross_validation_sets[0].items():
        print(f"Training fold {index + 1}/{num_folds}")

        model = fit(
            cross_validation["training"],
            data_token_indicator,
            save_to=None
        )
        models[index] = model

        validation_set = cross_validation["validation"]
        Validation = csr_matrix([
            np.concatenate([
                data_token_indicator(sentence1),
                data_token_indicator(sentence2)
            ])
            for sentence1, sentence2, label in validation_set
        ])
        y_validation = [label for _, _, label in validation_set]

        predictions = model.predict(Validation)
        accuracy = accuracy_score(y_validation, predictions)

        fold_results.append({
            "fold": index,
            "accuracy": accuracy
        })

        print(f"Validation accuracy: {accuracy:.4f}")

    print(f"\nMean accuracy: {np.mean([r['accuracy'] for r in fold_results]):.4f}")
    return fold_results, models
    


def cross_validation_train_matrix(X, labels, my_cross_validation_sets, num_folds):
    fold_results = []
    models = {}

    train_data = dataset["train"]

    if X.shape[0] != len(train_data):
        raise ValueError(
            f"X has {X.shape[0]} rows, but dataset['train'] has {len(train_data)} rows."
        )

    if len(labels) != X.shape[0]:
        raise ValueError(
            f"labels has {len(labels)} entries, but X has {X.shape[0]} rows."
        )

    # Map each sentence pair and label to its original row(s) in X.
    pair_to_indices = defaultdict(list)

    for row_index, (sentence1, sentence2, label) in enumerate(zip(
        train_data["sentence1"],
        train_data["sentence2"],
        train_data["label"]
    )):
        key = (sentence1, sentence2, label)
        pair_to_indices[key].append(row_index)

    for fold_index, split in my_cross_validation_sets[0].items():
        # Reset for every fold. Do not pop from pair_to_indices.
        used = defaultdict(int)

        def get_indices(examples):
            indices = []

            for sentence1, sentence2, label in examples:
                key = (sentence1, sentence2, label)
                matching_indices = pair_to_indices.get(key, [])
                occurrence = used[key]

                if occurrence >= len(matching_indices):
                    raise ValueError(
                        f"Could not match this example to a row in X: {key}"
                    )

                indices.append(matching_indices[occurrence])
                used[key] += 1

            return indices

        train_indices = get_indices(split["training"])
        validation_indices = get_indices(split["validation"])

        if not train_indices or not validation_indices:
            raise ValueError(
                f"Fold {fold_index} has an empty training or validation set."
            )

        y_train = [labels[i] for i in train_indices]
        y_validation = [labels[i] for i in validation_indices]

        model = LogisticRegression(
            solver="saga",
            max_iter=100,
            n_jobs=-1
        )

        model.fit(X[train_indices], y_train)
        predictions = model.predict(X[validation_indices])

        accuracy = accuracy_score(y_validation, predictions)

        models[fold_index] = model
        fold_results.append({
            "fold": fold_index,
            "accuracy": accuracy
        })

        print(f"Fold {fold_index + 1}/{num_folds}: {accuracy:.4f}")

    mean_accuracy = np.mean([result["accuracy"] for result in fold_results])
    print(f"Mean accuracy: {mean_accuracy:.4f}")

    return fold_results, models

labels = list(dataset["train"]["label"])

fold_results_bert, models_bert = cross_validation_train_matrix(
    X_bert,
    labels,
    my_cross_validation_sets,
    num_folds
)


fold_results_token_indicator, models_token_indicator = cross_validation_train_logistic(data_token_indicator, my_cross_validation_sets, num_folds)
fold_results_bag_of_words, models_bag_of_words = cross_validation_train_logistic(data_bag_of_words, my_cross_validation_sets, num_folds)


c:\Users\leonardo\miniconda3\envs\optimus\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Fold 1/5: 0.5941


c:\Users\leonardo\miniconda3\envs\optimus\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Fold 2/5: 0.5824


c:\Users\leonardo\miniconda3\envs\optimus\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Fold 3/5: 0.5915


... (20 outputs left out)

c:\Users\leonardo\miniconda3\envs\optimus\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Validation accuracy: 0.5025
Training fold 5/5


c:\Users\leonardo\miniconda3\envs\optimus\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Validation accuracy: 0.5047

Mean accuracy: 0.5062


# MLP
THE same thing but now with an MLP
Ones again the data loader from the given model does not work with our dataswet ass such i change it


In [16]:
from scipy.sparse import csr_matrix
from torch.utils.data import Dataset
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from tqdm.auto import tqdm
import time
import scipy
import scipy.sparse
import json
import numpy as np

class SparseMLP(nn.Module):
    """One-hidden-layer MLP that accepts sparse input tensors.

    Inputs: feature vectors of size `input_dim` (e.g. bag-of-words).
    Process: sparse matmul into a dense hidden layer, ReLU + dropout, linear (num_classes) and log-softmax outputs
        log-softmax, rather than softmax, for numerical stability
    Output: logprobs over `num_classes`.
    """

    def __init__(self, input_dim, hidden_dim, num_classes, dropout=0.2):
        """
        input_dim: size of the input feature space (e.g. vocabulary size)
        hidden_dim: width of the hidden layer
        num_classes: number of output labels
        dropout: fraction of hidden units zeroed during training (regularisation)
        """
        super().__init__()

        # First-layer weights stay dense; inputs arriving here are sparse.
        self.weight = nn.Parameter(
            torch.empty(input_dim, hidden_dim)
        )
        self.bias = nn.Parameter(
            torch.zeros(hidden_dim)
        )

        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

        # Everything after the first layer is dense.
        self.output = nn.Sequential(
            nn.Linear(hidden_dim, num_classes),            
            nn.LogSoftmax(dim=-1) # for numerical stability, it is common to use log softmax, rather than softmax
        )

        nn.init.xavier_uniform_(self.weight) # why xavier?

    def forward(self, x):
        """Map a sparse batch [batch, input_dim] to class log-probabilities [batch, num_classes]."""
        # First matmul keeps x sparse; result is dense from here on.
        x = torch.sparse.mm(x, self.weight)
        x = x + self.bias

        x = self.activation(x)
        x = self.dropout(x)
        # logprobs
        return self.output(x)

def scipy_to_torch_sparse(x):
    """Convert a SciPy sparse matrix to a coalesced PyTorch sparse COO tensor.

    Input: SciPy sparse matrix.
    Output: torch sparse tensor with the same shape and non-zero values.
    """
    x = x.tocoo()
    indices = torch.tensor([x.row, x.col], dtype=torch.long)
    values = torch.tensor(x.data, dtype=torch.float32)

    return torch.sparse_coo_tensor(indices, values, size=x.shape).coalesce()

class SparseDataset(Dataset):
    """Wraps a SciPy sparse feature matrix and label list for use with DataLoader.

    Inputs: X (sparse matrix of examples), y (integer labels).
    Output per item: one sparse row and its label tensor.
    """

    def __init__(self, X, y):
        self.X = X
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

def sparse_collate(batch):
    """Collate individual sparse rows into one PyTorch sparse batch.

    Input: list of (sparse row, label) pairs from SparseDataset.
    Process: stack SciPy rows, convert to a torch sparse tensor, stack labels.
    Output: (X_batch sparse tensor, y_batch long tensor).
    """
    X_rows, y_rows = zip(*batch)

    # Stack SciPy sparse rows, then convert the whole batch at once.
    X_batch = scipy.sparse.vstack(X_rows)
    X_batch = scipy_to_torch_sparse(X_batch)

    y_batch = torch.stack(y_rows)

    return X_batch, y_batch


def train_one_epoch(model, dataloader, optimizer, criterion, device="cpu"):
    """Run one training epoch: forward, loss, backward, optimizer step.

    Inputs: model, train dataloader, optimizer, loss function, device.
    Output: dict with average train loss and accuracy for the epoch.
    """
    if isinstance(device, str):
        device = torch.device(device)
    
    model.train()  # enable dropout / training behaviour

    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for X_batch, y_batch in dataloader:
        # move the batch to the device
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()  # clear gradients from the previous step

        logprobs = model(X_batch) # forward pass
        loss = criterion(logprobs, y_batch) # compute the loss

        loss.backward()   # compute gradients
        optimizer.step()  # update parameters

        # accumulate weighted loss and correct predictions for epoch averages
        total_loss += loss.item() * y_batch.size(0)
        predictions = logprobs.argmax(dim=1) # get the predicted class
        total_correct += (predictions == y_batch).sum().item() # count the number of correct predictions
        total_examples += y_batch.size(0) # count the number of examples

    return {
        "loss": total_loss / total_examples, # average loss
        "accuracy": total_correct / total_examples, # average accuracy
    }

def evaluate(model, dataloader, criterion, device="cpu"):
    """Evaluate the model without updating weights.

    Inputs: model, eval dataloader, loss function, device.
    Output: (metrics dict with loss/accuracy, numpy array of predicted labels).
    """
    if isinstance(device, str):
        device = torch.device(device)
        
    model.eval()  # disable dropout / training behaviour

    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    all_preds = []
    with torch.no_grad():  # no gradients needed for evaluation

        for X_batch, y_batch in dataloader:
            # move the batch to the device
            X_batch = X_batch.to(device) 
            y_batch = y_batch.to(device)

            logprobs = model(X_batch) # forward pass

            loss = criterion(logprobs, y_batch) # compute the loss

            # accumulate metrics; also keep predictions for later use
            total_loss += loss.item() * y_batch.size(0)
            predictions = logprobs.argmax(dim=1) # get the predicted class
            all_preds.append(predictions)

            total_correct += (predictions == y_batch).sum().item() # count the number of correct predictions
            total_examples += y_batch.size(0)

    # concatenate batch predictions into one array
    return {
        "loss": total_loss / total_examples,
        "accuracy": total_correct / total_examples
    }, torch.cat([t.detach().cpu() for t in all_preds]).numpy()

def train_neural_network_model(training_set, validation_set, num_classes, encoder, save_to,   
                                hidden_dim=128, dropout=0.1, batch_size=64, learning_rate=1e-3, 
                                weight_decay=1e-4, num_epochs=20, device=torch.device("cpu")):
    """Encode texts, train SparseMLP, and checkpoint the best validation model.

    Inputs: train/val (text, label) pairs, num_classes, encoder (text -> sparse vector), path prefix to save weights,
            plus standard training hyperparameters.
    Process: vectorise documents, build sparse dataloaders, train for `num_epochs`, save best-by-val-loss.
    Output: per-epoch metrics list, best epoch index, and validation predictions from the best model.
    """
    if isinstance(device, str):
        device = torch.device(device)
        
    time_start = time.time()
    # Convert the training and validation sets to vectors
    print("Transforming documents...")
    print(f"Training set size: {training_set[0][0:3]}")
    training_vectors = [np.concatenate([encoder(text[0]), encoder(text[1])]) for text in training_set]
    training_labels = [int(label[2]) for label in training_set]
    validation_vectors = [np.concatenate([encoder(text[0]), encoder(text[1])]) for text in validation_set]
    validation_labels = [int(label[2]) for label in validation_set]

    time_document_to_vector = time.time()
    print(f"Document transformation done! Time taken: {time_document_to_vector - time_start:.3f} seconds")
    
    # Keep SciPy sparse here; conversion to torch happens in sparse_collate
    X_train_sparse = csr_matrix(training_vectors)
    X_val_sparse = csr_matrix(validation_vectors)

    train_dataset = SparseDataset(X_train_sparse, training_labels) # create the training dataset
    val_dataset = SparseDataset(X_val_sparse, validation_labels) # create the validation dataset

    # create the data loaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=sparse_collate,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=sparse_collate,
    )

    time_dataloader = time.time()
    print(f"Dataloader done! Time taken: {time_dataloader - time_document_to_vector:.3f} seconds")

    # save architecture-defining arguments
    with open(f"{save_to}.json", "w", encoding="utf-8") as f:
        args = {
            'input_dim': 2*len(encoder), 
            'hidden_dim': hidden_dim, 
            'num_classes': num_classes, 
            'dropout': dropout,
        }
        json.dump(args, f, indent=2, ensure_ascii=False)
        
    # Create the model based on the input dimension, chosen hidden dimension, number of classification classes and dropout rate
    model = SparseMLP(input_dim=2*len(encoder), hidden_dim=hidden_dim, num_classes=num_classes, dropout=dropout).to(device)
    criterion = nn.NLLLoss() # create the loss function
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay) # create the optimizer

    metrics = []
    print("Training...")
    predictions = None
    for epoch in tqdm(range(num_epochs)):
        # Train the model for one epoch
        train_metrics = train_one_epoch(model, train_loader, optimizer, criterion, device)
        # Evaluate the model on the validation set
        val_metrics, preds = evaluate(model, val_loader, criterion, device)
        # Store the metrics for this epoch
        metrics.append({"train_metrics": train_metrics, "val_metrics": val_metrics})

        print(
            f"Epoch {epoch + 1:02d} | "
            f"train loss: {train_metrics['loss']:.4f} | "
            f"train acc: {train_metrics['accuracy']:.4f} | "
            f"val loss: {val_metrics['loss']:.4f} | "
            f"val acc: {val_metrics['accuracy']:.4f}"
            )
        
        # Saving the best model according to validation loss
        if epoch != 0:
            if val_metrics["loss"] <= best_val_loss:
                best_val_loss = val_metrics["loss"]
                torch.save(model.state_dict(), f"{save_to}.pt")
                best_epoch = epoch
                predictions = preds
        else:
            best_val_loss = val_metrics["loss"]
            torch.save(model.state_dict(), f"{save_to}.pt")
            best_epoch = epoch
            predictions = preds
    
    time_training = time.time()
    print(f"Training done! Time taken: {time_training - time_dataloader:.3f} seconds")

    return metrics, best_epoch, predictions


def load(path, device="cpu"):
    """
    Reconstruct the model architecture (using {path}.json) and load trained weights (using {path}.pt) and return it.
    """

    with open(f"{path}.json", "r", encoding="utf-8") as f:
        args = json.load(f)
    
    with open(f"{path}.pt", "rb") as f:
        state_dict = torch.load(f)
        
    if isinstance(device, str):
        device = torch.device(device)
        
    model = SparseMLP(**args)
    model.load_state_dict(state_dict)
    model = model.to(device)
    return model


def predict_probs(texts, model, encoder, batch_size=10):
    """
    Return a numpy array with shape [num_texts, num_classes] with the pmfs of the input texts as predicted by the model.
    
    texts: list of texts, each a string
    model: trained model
    encoder: encoder used in training
    batch_size: how many at a time
    """
    
    device = model.weight.device
        
    vectors = [encoder(x) for x in texts]
    labels = [0 for _ in texts]

    x_sparse = csr_matrix(vectors)

    dl = DataLoader(
        SparseDataset(x_sparse, labels),
        batch_size=batch_size,
        shuffle=False,
        collate_fn=sparse_collate,
    )
    
    model.eval()  # disable dropout / training behaviour
    
    all_probs = []
    with torch.no_grad():  # no gradients needed for evaluation

        for x_batch, y_batch in dl:
            # move the batch to the device
            x_batch = x_batch.to(device) 
            y_batch = y_batch.to(device)
            logprobs = model(x_batch) # forward pass
            # accumulate metrics; also keep predictions for later use
            all_probs.extend(logprobs.exp().detach().cpu().numpy())
            
    return np.array(all_probs)

# def cross_validation_train_neural_network(encoder, my_cross_validation_sets, num_folds):
#     """
#     Train a neural network model on each fold of the cross validation sets and evaluate it on the validation set.
#     """
#     smlp_results = []
#     for index, cross_validation in my_cross_validation_sets[0].items():
#         print(f"Training fold {index + 1}/{num_folds}")

#         training_set = cross_validation["training"]
#         validation_set = cross_validation["validation"]
#         num_classes = len(paws_outcome_space)
    
#         model = train_neural_network_model(
#             training_set,
#             validation_set,
#             num_classes,
#             encoder,
#             save_to=None
#         )

#         y_validation = [label for _, _, label in validation_set]
#         predictions = model.predict(validation_set)
#         accuracy = accuracy_score(y_validation, predictions)

#         smlp_results.append({
#             "fold": index,
#             "accuracy": accuracy
#         })

#         print(f"Validation accuracy: {accuracy:.4f}")
#     return smlp_results

def cross_validation_train_neural_network(encoder, my_cross_validation_sets, num_folds):
    smlp_results = []

    for index, cross_validation in my_cross_validation_sets[0].items():
        print(f"Training fold {index + 1}/{num_folds}")

        training_set = cross_validation["training"]
        validation_set = cross_validation["validation"]

        metrics, best_epoch, predictions = train_neural_network_model(
            training_set=training_set,
            validation_set=validation_set,
            num_classes=len(paws_outcome_space),
            encoder=encoder,
            save_to=f"mlp_fold_{index}"
        )

        y_validation = [label for _, _, label in validation_set]
        accuracy = accuracy_score(y_validation, predictions)

        smlp_results.append({
            "fold": index,
            "accuracy": accuracy,
            "best_epoch": best_epoch
        })

        print(f"Validation accuracy: {accuracy:.4f}")

    return smlp_results
from collections import defaultdict
from copy import deepcopy

def cross_validation_train_matrix_mlp(
    X,
    labels,
    my_cross_validation_sets,
    num_folds,
    num_classes=None,
    hidden_dim=128,
    dropout=0.1,
    batch_size=64,
    learning_rate=1e-3,
    weight_decay=1e-4,
    num_epochs=20,
    device="cpu",
):
    """
    Train and evaluate a SparseMLP on each fold using a precomputed
    feature matrix X. Assumes rows of X and labels match dataset["train"].
    """
    if isinstance(device, str):
        device = torch.device(device)

    X = scipy.sparse.csr_matrix(X, dtype=np.float32)
    labels = np.asarray(labels, dtype=np.int64)
    train_data = dataset["train"]

    if X.shape[0] != len(train_data) or len(labels) != X.shape[0]:
        raise ValueError(
            "X, labels, and dataset['train'] must have the same number of rows."
        )

    if num_classes is None:
        num_classes = int(labels.max()) + 1

    # Map each original example to its row in X.
    pair_to_indices = defaultdict(list)
    for row_index, (sentence1, sentence2, label) in enumerate(zip(
        train_data["sentence1"],
        train_data["sentence2"],
        train_data["label"],
    )):
        pair_to_indices[(sentence1, sentence2, label)].append(row_index)

    results = []
    models = {}

    for fold_index, split in my_cross_validation_sets[0].items():
        # Reset for every fold; do not remove indices from pair_to_indices.
        used = defaultdict(int)

        def get_indices(examples):
            indices = []

            for sentence1, sentence2, label in examples:
                key = (sentence1, sentence2, label)
                matches = pair_to_indices.get(key, [])
                occurrence = used[key]

                if occurrence >= len(matches):
                    raise ValueError(
                        f"Could not match fold example to a row in X: {key}"
                    )

                indices.append(matches[occurrence])
                used[key] += 1

            return indices

        train_indices = get_indices(split["training"])
        validation_indices = get_indices(split["validation"])

        if not train_indices or not validation_indices:
            raise ValueError(
                f"Fold {fold_index} has an empty training or validation set."
            )

        y_train = labels[train_indices].tolist()
        y_validation = labels[validation_indices].tolist()

        train_loader = DataLoader(
            SparseDataset(X[train_indices], y_train),
            batch_size=batch_size,
            shuffle=True,
            collate_fn=sparse_collate,
        )
        validation_loader = DataLoader(
            SparseDataset(X[validation_indices], y_validation),
            batch_size=batch_size,
            shuffle=False,
            collate_fn=sparse_collate,
        )

        model = SparseMLP(
            input_dim=X.shape[1],
            hidden_dim=hidden_dim,
            num_classes=num_classes,
            dropout=dropout,
        ).to(device)

        criterion = nn.NLLLoss()
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=learning_rate,
            weight_decay=weight_decay,
        )

        best_val_loss = float("inf")
        best_state = None
        best_accuracy = None
        best_epoch = None

        for epoch in range(num_epochs):
            train_one_epoch(model, train_loader, optimizer, criterion, device)
            val_metrics, _ = evaluate(
                model, validation_loader, criterion, device
            )

            if val_metrics["loss"] < best_val_loss:
                best_val_loss = val_metrics["loss"]
                best_accuracy = val_metrics["accuracy"]
                best_epoch = epoch
                best_state = deepcopy(model.state_dict())

        model.load_state_dict(best_state)
        models[fold_index] = model

        results.append({
            "fold": fold_index,
            "accuracy": best_accuracy,
            "best_epoch": best_epoch,
        })

        print(
            f"Fold {fold_index + 1}/{num_folds}: "
            f"{best_accuracy:.4f} (best epoch {best_epoch + 1})"
        )

    mean_accuracy = np.mean([result["accuracy"] for result in results])
    print(f"Mean accuracy: {mean_accuracy:.4f}")

    return results, models


labels = list(dataset["train"]["label"])

results_bert, models_bert = cross_validation_train_matrix_mlp(
    X_bert, labels, my_cross_validation_sets, num_folds,
    num_classes=len(paws_outcome_space), device=device,
)


cross_validation_results_token_indicator = cross_validation_train_neural_network(
    data_token_indicator,
    my_cross_validation_sets,
    num_folds
)
cross_validation_results_bag_of_words = cross_validation_train_neural_network(
    data_bag_of_words,
    my_cross_validation_sets,
    num_folds
)



Fold 1/5: 0.5984 (best epoch 19)
Fold 2/5: 0.5933 (best epoch 19)


KeyboardInterrupt: 

In [ ]:
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

class DenseMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
            nn.LogSoftmax(dim=-1),
        )
    def forward(self, x):
        return self.net(x)

def make_loader(X, y, batch_size, shuffle):
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32), 
                        torch.tensor(y, dtype=torch.long))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)
y_all = np.array(dataset["train"]["label"])

bert_mlp_results = []

for index, fold in my_cross_validation_sets[0].items():
    tr = np.asarray(fold["training_ids"], dtype=int)
    va = np.asarray(fold["validation_ids"], dtype=int)

    X_tr , y_tr = X_bert[tr], y_all[tr]
    X_va, y_va = X_bert[va], y_all[va]
    scaler = StandardScaler().fit(X_tr)
    X_tr, X_va = scaler.transform(X_tr), scaler.transform(X_va)
    train_loader = make_loader(X_tr, y_tr, batch_size=64, shuffle=True)
    val_loader = make_loader(X_va, y_va, batch_size=256, shuffle=False)

    model = DenseMLP(X_tr.shape[1], hidden_dim=128, num_classes=len(paws_outcome_space), dropout=0.1).to(device)
    criterion = nn.NLLLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    
    best_loss, best_preds = float("inf"), None
    for epoch in range(20):
        train_metrics = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_metrics, preds = evaluate(model, val_loader, criterion, device)
        # print(f"  epoch {epoch+1:02d} | train loss {train_metrics['loss']:.4f} acc {train_metrics['accuracy']:.4f} "
        #       f"| val loss {val_metrics['loss']:.4f} acc {val_metrics['accuracy']:.4f}")
        if val_metrics["loss"] < best_loss:
            best_loss, best_preds = val_metrics["loss"], preds
    acc = accuracy_score(y_va, best_preds)
    bert_mlp_results.append({"fold": index, "accuracy": acc})
    print(f"Fold {index + 1}: {acc:.4f}")
print(f"Mean accuracy: {np.mean([r['accuracy'] for r in bert_mlp_results]):.4f}")






  epoch 01 | train loss 0.6819 acc 0.5680 | val loss 0.6711 acc 0.5897
  epoch 02 | train loss 0.6633 acc 0.5947 | val loss 0.6609 acc 0.5949
  epoch 03 | train loss 0.6521 acc 0.6052 | val loss 0.6621 acc 0.5892
  epoch 04 | train loss 0.6440 acc 0.6129 | val loss 0.6597 acc 0.6004
  epoch 05 | train loss 0.6371 acc 0.6198 | val loss 0.6573 acc 0.6001
  epoch 06 | train loss 0.6322 acc 0.6234 | val loss 0.6595 acc 0.6078
  epoch 07 | train loss 0.6269 acc 0.6277 | val loss 0.6536 acc 0.6053
  epoch 08 | train loss 0.6229 acc 0.6314 | val loss 0.6673 acc 0.6003
  epoch 09 | train loss 0.6179 acc 0.6341 | val loss 0.6642 acc 0.6053
  epoch 10 | train loss 0.6142 acc 0.6376 | val loss 0.6707 acc 0.5971
  epoch 11 | train loss 0.6123 acc 0.6415 | val loss 0.6677 acc 0.6034
  epoch 12 | train loss 0.6089 acc 0.6426 | val loss 0.6806 acc 0.6027
  epoch 13 | train loss 0.6048 acc 0.6438 | val loss 0.6902 acc 0.6027
  epoch 14 | train loss 0.6010 acc 0.6477 | val loss 0.7020 acc 0.5832
  epoc

In [ ]:
from copy import deepcopy

def cross_validation_train_encoded_mlp(
    encoder,
    my_cross_validation_sets,
    num_folds,
    num_classes=None,
    hidden_dim=128,
    dropout=0.1,
    batch_size=64,
    learning_rate=1e-3,
    weight_decay=1e-4,
    num_epochs=20,
    device="cpu",
):
    """
    Train a SparseMLP on sentence-pair features generated by a callable encoder.
    The encoder must map one sentence to a fixed-size vector.
    """

    if isinstance(device, str):
        device = torch.device(device)

    def as_sparse_row(vector):
        # Support encoders that return PyTorch tensors.
        if isinstance(vector, torch.Tensor):
            vector = vector.detach().cpu().numpy()

        # Preserve sparse outputs without densifying them.
        if scipy.sparse.issparse(vector):
            row = vector.tocsr()

            if row.shape[0] == 1:
                return row.astype(np.float32)
            if row.shape[1] == 1:
                return row.T.tocsr().astype(np.float32)

            raise ValueError(
                f"Encoder output must be a vector or a single row; got {row.shape}."
            )

        # Convert dense vector output to a one-row sparse matrix.
        vector = np.asarray(vector, dtype=np.float32).reshape(1, -1)
        return csr_matrix(vector)

    def encode_pairs(examples):
        rows = []

        for sentence1, sentence2, _ in examples:
            vector1 = as_sparse_row(encoder(sentence1))
            vector2 = as_sparse_row(encoder(sentence2))

            if vector1.shape[1] != vector2.shape[1]:
                raise ValueError(
                    "The encoder returned different vector sizes "
                    "for the two sentences in a pair."
                )

            rows.append(
                scipy.sparse.hstack([vector1, vector2], format="csr")
            )

        if not rows:
            raise ValueError("Cannot encode an empty set of examples.")

        return scipy.sparse.vstack(rows, format="csr", dtype=np.float32)

    results = []
    models = {}

    for fold_index, split in my_cross_validation_sets[0].items():
        print(f"Training fold {fold_index + 1}/{num_folds}")

        training_set = split["training"]
        validation_set = split["validation"]

        if not training_set or not validation_set:
            raise ValueError(
                f"Fold {fold_index} has an empty training or validation set."
            )

        X_train = encode_pairs(training_set)
        X_validation = encode_pairs(validation_set)

        if X_train.shape[1] != X_validation.shape[1]:
            raise ValueError(
                "Training and validation feature dimensions do not match."
            )

        y_train = [int(example[2]) for example in training_set]
        y_validation = [int(example[2]) for example in validation_set]

        if num_classes is None:
            fold_num_classes = max(y_train + y_validation) + 1
        else:
            fold_num_classes = num_classes

        train_loader = DataLoader(
            SparseDataset(X_train, y_train),
            batch_size=batch_size,
            shuffle=True,
            collate_fn=sparse_collate,
        )
        validation_loader = DataLoader(
            SparseDataset(X_validation, y_validation),
            batch_size=batch_size,
            shuffle=False,
            collate_fn=sparse_collate,
        )

        model = SparseMLP(
            input_dim=X_train.shape[1],
            hidden_dim=hidden_dim,
            num_classes=fold_num_classes,
            dropout=dropout,
        ).to(device)

        criterion = nn.NLLLoss()
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=learning_rate,
            weight_decay=weight_decay,
        )

        best_val_loss = float("inf")
        best_state = None
        best_accuracy = None
        best_epoch = None

        for epoch in range(num_epochs):
            train_one_epoch(model, train_loader, optimizer, criterion, device)
            val_metrics, _ = evaluate(
                model, validation_loader, criterion, device
            )

            if val_metrics["loss"] < best_val_loss:
                best_val_loss = val_metrics["loss"]
                best_accuracy = val_metrics["accuracy"]
                best_epoch = epoch
                best_state = deepcopy(model.state_dict())

        model.load_state_dict(best_state)
        models[fold_index] = model

        results.append({
            "fold": fold_index,
            "accuracy": best_accuracy,
            "best_epoch": best_epoch,
        })

        print(
            f"Validation accuracy: {best_accuracy:.4f} "
            f"(best epoch {best_epoch + 1})"
        )

    print(f"Mean accuracy: {np.mean([r['accuracy'] for r in results]):.4f}")
    return results, models
results_token_indicator, models_token_indicator = cross_validation_train_encoded_mlp(
    data_token_indicator, my_cross_validation_sets, num_folds,
    num_classes=len(paws_outcome_space), device=device,
)
results_bag_of_words, models_bag_of_words = cross_validation_train_encoded_mlp(
    data_bag_of_words, my_cross_validation_sets, num_folds,
    num_classes=len(paws_outcome_space), device=device,
)


Training fold 1/5


KeyboardInterrupt: 

In [ ]:
# Numbers and figures used in the oral slides.
# Needs these from the cells above: part1, dataset, tokenizer, my_cross_validation_sets.
# After the full run: update fold_scores in section 5 and re-run this cell.
import json
import random
import re
import subprocess
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np

TRAIN_COLOUR, VALIDATION_COLOUR = "#2E5FA3", "#B84A12"
ROPE = 0.005  # about one standard error of accuracy on a fold of 9,880 pairs

s1 = list(dataset["train"]["sentence1"])
s2 = list(dataset["train"]["sentence2"])
y = np.array(dataset["train"]["label"])
N = len(y)
folds = my_cross_validation_sets[0]
fold_ids = [(np.asarray(f["training_ids"], dtype=int), np.asarray(f["validation_ids"], dtype=int))
            for f in folds.values()]


# 1. Slide 1: word overlap by label (lower-cased, split on spaces)
jaccard = np.zeros(N)
same_bag = np.zeros(N, dtype=bool)
for i in range(N):
    words1, words2 = s1[i].lower().split(), s2[i].lower().split()
    set1, set2 = set(words1), set(words2)
    jaccard[i] = len(set1 & set2) / len(set1 | set2)
    same_bag[i] = Counter(words1) == Counter(words2)

print("Slide 1: dataset")
print(f"  exact duplicate pairs: {N - len(set(zip(s1, s2)))}")
for label, name in enumerate(["not paraphrase", "paraphrase"]):
    mask = y == label
    print(f"  {name}: share {mask.mean():.1%}, overlap {jaccard[mask].mean():.3f}, identical word bag {same_bag[mask].mean():.1%}")
print(f"  pairs with an identical word bag that are paraphrases: {y[same_bag].mean():.1%}")

bins = np.linspace(0.6, 1.0, 21)
centres = (bins[:-1] + bins[1:]) / 2
fig, ax = plt.subplots(figsize=(8, 3.2))
for label, name, colour in ((0, "Not paraphrase", VALIDATION_COLOUR), (1, "Paraphrase", TRAIN_COLOUR)):
    counts, _ = np.histogram(np.clip(jaccard[y == label], 0.6, 1.0), bins=bins)
    ax.plot(centres, 100 * counts / (y == label).sum(), color=colour, linewidth=2, label=name)
ax.set_xlabel("word overlap (Jaccard); values below 0.6 are counted in the first bin")
ax.set_ylabel("% of pairs with that label")
ax.set_title("Word overlap does not separate the labels", loc="left", fontsize=11)
ax.grid(axis="y", color="#E5E7EB")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False)
fig.tight_layout()
plt.show()


# 2. Slides 1 and 2: BPE token counts, on a sample of 8,000 pairs
sample = random.Random(0).sample(range(N), 8000)
encoded = [tokenizer.encode(s1[i]).ids for i in sample] + [tokenizer.encode(s2[i]).ids for i in sample]
total_tokens = sum(len(ids) for ids in encoded)
unique_tokens = sum(len(set(ids)) for ids in encoded)
print("\nSlides 1 and 2: tokens")
print(f"  tokens per sentence: {total_tokens / len(encoded):.1f}, unique: {unique_tokens / len(encoded):.1f}")
print(f"  mean count of a token that occurs: {total_tokens / unique_tokens:.2f}")


# 3. Slides 2 and 4: majority baseline and word-overlap threshold, per fold
def overlap_threshold_accuracy(train_ids, validation_ids):
    """Pick the threshold and direction on the training fold, score on the validation fold."""
    best_accuracy, best_threshold, best_direction = -1, 0.0, True
    for threshold in np.linspace(0, 1, 101):
        for direction in (True, False):
            predictions = (jaccard[train_ids] >= threshold) == direction
            accuracy = (predictions == y[train_ids]).mean()
            if accuracy > best_accuracy:
                best_accuracy, best_threshold, best_direction = accuracy, threshold, direction
    predictions = (jaccard[validation_ids] >= best_threshold) == best_direction
    return (predictions == y[validation_ids]).mean()

baseline_folds = []
overlap_folds = []
for train_ids, validation_ids in fold_ids:
    majority_label = Counter(y[train_ids].tolist()).most_common(1)[0][0]
    baseline_folds.append(float((y[validation_ids] == majority_label).mean()))
    overlap_folds.append(float(overlap_threshold_accuracy(train_ids, validation_ids)))

print("\nSlides 2 and 4: reference scores")
print(f"  majority baseline: {np.mean(baseline_folds):.4f} +- {np.std(baseline_folds, ddof=1):.4f}, folds {np.round(baseline_folds, 4)}")
print(f"  overlap threshold: {np.mean(overlap_folds):.4f}, folds {np.round(overlap_folds, 4)}")


# 4. Slide 5: sentences shared between training and validation folds
print("\nSlide 5: repeated sentences")
for fold, (train_ids, validation_ids) in enumerate(fold_ids):
    train_sentences = {s1[i] for i in train_ids} | {s2[i] for i in train_ids}
    any_shared = np.mean([s1[i] in train_sentences or s2[i] in train_sentences for i in validation_ids])
    both_shared = np.mean([s1[i] in train_sentences and s2[i] in train_sentences for i in validation_ids])
    labels_by_first = {}
    for i in train_ids:
        labels_by_first.setdefault(s1[i], []).append(int(y[i]))
    seen = [i for i in validation_ids if s1[i] in labels_by_first]
    copied = np.mean([Counter(labels_by_first[s1[i]]).most_common(1)[0][0] == y[i] for i in seen])
    print(f"  fold {fold}: any sentence {any_shared:.1%}, both {both_shared:.1%}, "
          f"first sentence {len(seen) / len(validation_ids):.1%}, copied label accuracy {copied:.1%}")

first_counts = Counter(s1)
labels_of_first = {}
for sentence, label in zip(s1, y.tolist()):
    labels_of_first.setdefault(sentence, set()).add(label)
repeated = [sentence for sentence, count in first_counts.items() if count > 1]
print(f"  repeated first sentences: {len(repeated)}, with both labels: {sum(len(labels_of_first[s]) == 2 for s in repeated)}")


# 5. Slide 4: fold accuracies, Bayesian comparison and posterior plot
# Sources: the three log-linear rows are the output of cell 20. Do not use the "Mean accuracy" it
# prints for bag-of-words (0.5062): fold_results is global there, so it also averages the indicator
# folds. The six MLP rows come from run_mlps.py, saved in results/mlp_fold_scores.json.
fold_scores = {
    "Log-linear\nindicator": [0.5069, 0.5044, 0.5137, 0.5018, 0.5004],
    "Log-linear\nBoW": [0.5116, 0.5039, 0.5118, 0.5025, 0.5047],
    "Log-linear\nBERT": [0.5941, 0.5824, 0.5915, 0.5912, 0.5928],
    "MLP\nindicator": [0.5963, 0.5865, 0.5888, 0.5895, 0.6021],
    "MLP\nBoW": [0.5952, 0.5842, 0.5810, 0.5922, 0.6019],
    "MLP\nBERT": [0.5888, 0.5727, 0.5869, 0.5848, 0.5796],
    "Two-layer\nindicator": [0.6057, 0.5989, 0.6004, 0.6088, 0.6007],
    "Two-layer\nBoW": [0.6074, 0.6076, 0.6008, 0.6095, 0.5992],
    "Two-layer\nBERT": [0.6044, 0.6040, 0.6019, 0.6066, 0.6013],
}

print("\nSlide 4: accuracy %")
for name, scores in fold_scores.items():
    spread = f" +- {100 * np.std(scores, ddof=1):.1f}" if len(scores) > 1 else ""
    print(f"  {name.replace(chr(10), ' ')}: {100 * np.mean(scores):.1f}{spread} ({len(scores)}/5 folds)")

comparisons = [
    ("Baseline", baseline_folds, "Two-layer BoW", fold_scores["Two-layer\nBoW"]),
    ("Two-layer BERT", fold_scores["Two-layer\nBERT"], "Two-layer BoW", fold_scores["Two-layer\nBoW"]),
    ("Two-layer indicator", fold_scores["Two-layer\nindicator"], "Two-layer BoW", fold_scores["Two-layer\nBoW"]),
    ("Log-linear BERT", fold_scores["Log-linear\nBERT"], "Two-layer BERT", fold_scores["Two-layer\nBERT"]),
    ("MLP BERT", fold_scores["MLP\nBERT"], "Two-layer BERT", fold_scores["Two-layer\nBERT"]),
    ("MLP BoW", fold_scores["MLP\nBoW"], "Two-layer BoW", fold_scores["Two-layer\nBoW"]),
    ("MLP indicator", fold_scores["MLP\nindicator"], "Two-layer indicator", fold_scores["Two-layer\nindicator"]),
    ("Log-linear BoW", fold_scores["Log-linear\nBoW"], "MLP BoW", fold_scores["MLP\nBoW"]),
    ("Log-linear indicator", fold_scores["Log-linear\nindicator"], "Log-linear BoW", fold_scores["Log-linear\nBoW"]),
]
print(f"\nSlide 4: Bayesian correlated t-test, ROPE +-{ROPE}")
for name_a, scores_a, name_b, scores_b in comparisons:
    posterior = part1.analysis.BenavoliCorrelatedT(scores_a, scores_b, runs=1, names=[name_a, name_b])
    p_a, p_rope, p_b = posterior.probs(ROPE)
    print(f"  A={name_a}, B={name_b}: B-A {100 * posterior.mean_diff:+.1f} points, "
          f"P(A) {p_a:.4f}, P(ROPE) {p_rope:.4f}, P(B) {p_b:.4f}")

# Posterior plot for the two highest systems: A = Two-layer BERT, B = Two-layer BoW
closest = part1.analysis.BenavoliCorrelatedT(fold_scores["Two-layer\nBERT"], fold_scores["Two-layer\nBoW"],
                                             runs=1, names=["Two-layer BERT", "Two-layer BoW"])
print(f"  closest comparison, 95% interval for B-A: {np.round(100 * np.array(closest.mean_diff + np.array([-2.776, 2.776]) * closest.std), 1)} points")
fig, ax = closest.plot(ROPE)
plt.show()


# 6. Slides 3 and 5: train and validation accuracy per epoch
# These epoch logs are saved in commit 3ee4c78, because the current cells no longer print them.
def logged_output(commit="3ee4c78"):
    """Return all printed output stored in project.ipynb at that commit."""
    shown = subprocess.run(["git", "show", f"{commit}:./project.ipynb"], capture_output=True,
                           encoding="utf-8", errors="replace", check=True)
    notebook = json.loads(shown.stdout)
    return "\n".join("".join(output.get("text", [])) for cell in notebook["cells"]
                     for output in cell.get("outputs", []) if output.get("output_type") == "stream")

log = logged_output()
two_layer_rows = re.findall(r"epoch \d+ \| train loss [\d.]+ acc ([\d.]+) \| val loss [\d.]+ acc ([\d.]+)", log)
bow_rows = re.findall(r"Epoch \d+ \| train loss: [\d.]+ \| train acc: ([\d.]+) \| val loss: [\d.]+ \| val acc: ([\d.]+)", log)
assert len(two_layer_rows) == 100, f"expected 5 folds of 20 epochs, found {len(two_layer_rows)} rows"
assert len(bow_rows) == 20, f"expected 1 fold of 20 epochs, found {len(bow_rows)} rows"

two_layer = 100 * np.array(two_layer_rows, dtype=float).reshape(5, 20, 2).mean(axis=0)  # mean of 5 folds
bow = 100 * np.array(bow_rows, dtype=float)  # fold 1 only

def plot_epochs(curves, title, y_limits):
    epochs = np.arange(1, 21)
    fig, ax = plt.subplots(figsize=(7, 4.2))
    ax.plot(epochs, curves[:, 0], color=TRAIN_COLOUR, linewidth=2, label=f"Train, ends at {curves[-1, 0]:.1f}%")
    ax.plot(epochs, curves[:, 1], color=VALIDATION_COLOUR, linewidth=2, label=f"Validation, ends at {curves[-1, 1]:.1f}%")
    ax.axhline(100 * np.mean(baseline_folds), color="#374151", linestyle="--", linewidth=1.5,
               label=f"Majority baseline {100 * np.mean(baseline_folds):.1f}%")
    ax.set_xlim(1, 20)
    ax.set_xticks([1, 5, 10, 15, 20])
    ax.set_ylim(*y_limits)
    ax.set_xlabel("epoch")
    ax.set_ylabel("accuracy (%)")
    ax.set_title(title, loc="left", fontsize=11)
    ax.grid(axis="y", color="#E5E7EB")
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False)
    fig.tight_layout()
    plt.show()

plot_epochs(two_layer, "Two-layer MLP on BERT: accuracy per epoch, mean of 5 folds", (55, 70))
plot_epochs(bow, "MLP on bag-of-words: accuracy per epoch, fold 1", (45, 75))